# 03 · Regresión multivariable: consumo de energía


**Ruta sugerida:** lee la teoría, ejecuta las celdas en orden, predice lo que ocurrirá, realiza el ejercicio y luego compara con la solución. Puedes modificar y volver a ejecutar las celdas. Los datos son **sintéticos**: ilustran un contexto, no representan observaciones reales.

**Conceptos compartidos.** Una neurona calcula una combinación de entradas y un sesgo; una activación puede transformar el resultado. Durante `fit`, el optimizador ajusta los pesos para reducir una pérdida mediante gradientes. Una época es una pasada por los datos de entrenamiento. Validación ayuda a decidir cambios; prueba se reserva para la evaluación final. El descenso de entrenamiento con validación creciente puede indicar sobreajuste; ambas pérdidas altas pueden indicar subajuste. Estas son señales, no diagnósticos automáticos.

**Preparación:** Google Colab ya suele incluir las dependencias. En Jupyter local instala en el entorno activo `pip install tensorflow numpy matplotlib scikit-learn`; reinicia el kernel si acabas de instalar. Ejecuta todas las celdas en orden. El entrenamiento usa CPU y puede tardar unos minutos según el equipo.


**Tiempo orientativo:** 25–35 minutos.

## 1. Idea central

Una neurona puede combinar varias características: $\hat y=w_1x_1+w_2x_2+b$. Aquí estimamos consumo a partir de área y horas de uso. Ajustamos la escala solo con entrenamiento para evitar fuga de información.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

keras.utils.set_random_seed(42)
np.random.seed(42)
print("TensorFlow:", tf.__version__)

## 2. Datos y separación

Primero observa las formas: cada fila es un ejemplo y cada columna una característica. Reservamos 60 % para entrenamiento, 20 % para validación y 20 % para prueba.

In [ ]:
rng = np.random.default_rng(42)
area = rng.uniform(30, 150, 800)
horas = rng.uniform(1, 12, 800)
X = np.column_stack([area, horas]).astype("float32")
y = (20 + 0.35 * area + 5 * horas + rng.normal(0, 5, len(X))).astype("float32")

In [ ]:
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42
)
escalador = StandardScaler().fit(X_train)
X_train_s = escalador.transform(X_train).astype("float32")
X_val_s = escalador.transform(X_val).astype("float32")
X_test_s = escalador.transform(X_test).astype("float32")
print(
    "Formas entrenamiento / validación / prueba:",
    X_train.shape,
    X_val.shape,
    X_test.shape,
)

## 3. Modelo y entrenamiento

Antes de ejecutar: ¿qué esperas que ocurra con la pérdida de entrenamiento y de validación?

In [ ]:
modelo = keras.Sequential([keras.Input(shape=(2,)), keras.layers.Dense(1)])
modelo.compile(optimizer=keras.optimizers.Adam(0.1), loss="mse", metrics=["mae"])
modelo.summary()
historia = modelo.fit(
    X_train_s,
    y_train,
    validation_data=(X_val_s, y_val),
    epochs=120,
    batch_size=32,
    verbose=0,
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(historia.history["loss"], label="Entrenamiento")
plt.plot(historia.history["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Conversación:** ¿bajan ambas curvas? ¿Cuándo se estabilizan? ¿Hay separación entre ellas? Recuerda que la pérdida de validación puede fluctuar por el tamaño de la muestra.

## 4. Primera evaluación

Esta es una evaluación ilustrativa del modelo inicial. Si lo modificas a partir de estos resultados, el conjunto de prueba deja de ser una evaluación final independiente. Usa validación para elegir y conserva otra prueba final si ya la consultaste.

In [ ]:
resultado = dict(
    zip(modelo.metrics_names, modelo.evaluate(X_test_s, y_test, verbose=0))
)
print("Evaluación de prueba:", resultado)
pred = modelo.predict(X_test_s, verbose=0).ravel()

In [ ]:
plt.scatter(y_test, pred, alpha=0.5)
lim = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
plt.plot(lim, lim, "r--")
plt.xlabel("Consumo observado")
plt.ylabel("Consumo predicho")
plt.show()

## 5. Tu turno

Entrena otro modelo usando solo el área (primera columna). Compara su MAE de validación con el modelo de dos entradas. ¿Qué aporta conocer las horas?

Escribe tu código en la siguiente celda y responde en la siguiente celda de texto. Luego consulta la solución.

In [ ]:
# Tu código aquí

**Mi interpretación:** escribe aquí tu respuesta.

## 6. Solución de referencia

Intenta resolverlo antes de ejecutar la celda. Los resultados pueden variar ligeramente entre versiones y dispositivos.

In [ ]:
solo_area = keras.Sequential([keras.Input(shape=(1,)), keras.layers.Dense(1)])
solo_area.compile(optimizer=keras.optimizers.Adam(0.1), loss="mse", metrics=["mae"])
solo_area.fit(
    X_train_s[:, :1],
    y_train,
    validation_data=(X_val_s[:, :1], y_val),
    epochs=120,
    batch_size=32,
    verbose=0,
)
print(
    "MAE validación, solo área:",
    round(solo_area.evaluate(X_val_s[:, :1], y_val, verbose=0)[1], 3),
)
print(
    "MAE validación, ambas variables:",
    round(modelo.evaluate(X_val_s, y_val, verbose=0)[1], 3),
)

Las horas afectan el objetivo en la generación de datos, por lo que omitirlas normalmente aumenta el error. La importancia de una variable no debe deducirse directamente de los pesos sin considerar sus escalas y correlaciones.

## 7. Desafío opcional

Recupera los coeficientes en las unidades originales dividiendo cada peso por su desviación estándar de entrenamiento.

**Cierre:** menciona una conclusión de las curvas y una limitación de estos datos sintéticos.